# Feature-Name Anonymization Ablation — WUSTL-IIoT

Investigates whether GPT-4o selects features based on **semantic name reasoning**
(e.g. `SrcRate`, `DstPkts` sound attack-relevant) or **genuine statistical patterns**.

**Design:**
- Two conditions on identical data:
  - **Original**: real feature names (`SrcRate`, `TotBytes`, ...)
  - **Anonymized**: opaque labels `f0`, `f1`, ..., `f{n}`
- Representative samples fixed at Cell 3 (cosine-similarity to mean row).
- 3 independent seeds (temperature=0.1).

**Dataset:** `wustl-iiot-population.csv` — 1,107,448 normal vs 87,016 attack rows.
Balanced to 87,016 each → 174,032 total → 80/20 split → **~69,613/~69,613 train, ~17,403/~17,403 test**.

**Note:** `SrcAddr`, `DstAddr`, `StartTime`, `LastTime` are string columns kept as features.
The dtype-aware evaluate_rule handles string vs numeric comparisons correctly.

**Metrics:** ΔF1, Jaccard feature-selection overlap, semantic category distribution.

## Cell 1 — Load Dataset

In [1]:
################################################################################
# Load Dataset
#
# Source: wustl-iiot-population.csv
# Label column: 'Target' (int: 0=normal, 1=attack); also drop 'Traffic'
# Features: 46 columns after dropping 2 label columns
#
# Class imbalance: 1,107,448 normal vs 87,016 attack.
# Balance: sample normal down to match attack count.
# Then 80/20 split → ~69,613/~69,613 train, ~17,403/~17,403 test.
################################################################################

import pandas as pd
import os
import numpy as np
from sklearn.model_selection import train_test_split
from tabulate import tabulate

dataset_name = "wustl-iiot"

data_path = os.path.join(
    os.path.expanduser("~"),
    "Documents", "Projects", "RAG Paper",
    "data", "wustl-iiot", "wustl-iiot-population.csv"
)

df = pd.read_csv(data_path, low_memory=False)
print(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
print(f"Target value counts:\n{df['Target'].value_counts()}")

# Split by binary label
normal_all = df[df['Target'] == 0].drop(columns=['Target', 'Traffic'])
attack_all  = df[df['Target'] == 1].drop(columns=['Target', 'Traffic'])

# Balance: sample normal down to attack count (attack is minority)
normal_bal = normal_all.sample(n=len(attack_all), random_state=42).reset_index(drop=True)
attack_bal = attack_all.reset_index(drop=True)

# 80/20 split
normal_df_train, normal_df_test = train_test_split(normal_bal, test_size=0.2, random_state=42)
attack_df_train, attack_df_test = train_test_split(attack_bal, test_size=0.2, random_state=42)

normal_df_train = normal_df_train.reset_index(drop=True)
normal_df_test  = normal_df_test.reset_index(drop=True)
attack_df_train = attack_df_train.reset_index(drop=True)
attack_df_test  = attack_df_test.reset_index(drop=True)

data = [
    ["Normal", len(normal_df_train) + len(normal_df_test),
               len(normal_df_train), len(normal_df_test)],
    ["Attack", len(attack_df_train) + len(attack_df_test),
               len(attack_df_train), len(attack_df_test)],
]
print()
print(tabulate(data, headers=["Class", "Total", "Train", "Test"], tablefmt="grid"))

Loaded 1,194,464 rows, 49 columns
Target value counts:
Target
0    1107448
1      87016
Name: count, dtype: int64

+---------+---------+---------+--------+
| Class   |   Total |   Train |   Test |
+=========+=========+=========+========+
| Normal  |   87016 |   69612 |  17404 |
+---------+---------+---------+--------+
| Attack  |   87016 |   69612 |  17404 |
+---------+---------+---------+--------+


## Cell 2 — Anonymization Map + Semantic Categories

In [2]:
################################################################################
# Build anonymization maps and define semantic categories for WUSTL-IIoT
#
# 46 features after dropping Target and Traffic.
# String columns: StartTime, LastTime, SrcAddr, DstAddr.
# All others are numeric (int64 or float64).
################################################################################

feature_names = normal_df_train.columns.to_list()

anon_map    = {name: f"f{i}" for i, name in enumerate(feature_names)}
reverse_map = {f"f{i}": name for i, name in enumerate(feature_names)}

SEMANTIC_CATEGORIES = {
    "identity":      ["StartTime", "LastTime", "SrcAddr", "DstAddr"],
    "network":       ["Proto", "Sport", "Dport", "sTtl", "dTtl", "sTos",
                      "sIpId", "dIpId", "sDSb"],
    "packet_counts": ["SrcPkts", "DstPkts", "TotPkts", "SrcLoss", "DstLoss", "Loss"],
    "byte_counts":   ["SrcBytes", "DstBytes", "TotBytes",
                      "SAppBytes", "DAppBytes", "TotAppByte"],
    "rates":         ["SrcRate", "DstRate", "Load", "SrcLoad", "DstLoad", "pLoss"],
    "timing":        ["Dur", "IdleTime", "SIntPkt", "DIntPkt", "SrcJitter", "DstJitter",
                      "SrcJitAct", "DstJitAct", "TcpRtt", "SynAck", "RunTime"],
    "statistical":   ["Mean", "Sum", "Min", "Max"],
}

def get_category(feature_name: str) -> str:
    real = reverse_map.get(feature_name, feature_name)
    for cat, members in SEMANTIC_CATEGORIES.items():
        if real in members:
            return cat
    return "unknown"

print(f"Feature count: {len(feature_names)}")
print("\nAnonymization map:")
for k, v in anon_map.items():
    print(f"  {k:20s} → {v}")

print("\nSemantic categories:")
for cat, members in SEMANTIC_CATEGORIES.items():
    covered = [m for m in members if m in feature_names]
    print(f"  {cat:15s}: {covered}")

Feature count: 47

Anonymization map:
  StartTime            → f0
  LastTime             → f1
  SrcAddr              → f2
  DstAddr              → f3
  Mean                 → f4
  Sport                → f5
  Dport                → f6
  SrcPkts              → f7
  DstPkts              → f8
  TotPkts              → f9
  DstBytes             → f10
  SrcBytes             → f11
  TotBytes             → f12
  SrcLoad              → f13
  DstLoad              → f14
  Load                 → f15
  SrcRate              → f16
  DstRate              → f17
  Rate                 → f18
  SrcLoss              → f19
  DstLoss              → f20
  Loss                 → f21
  pLoss                → f22
  SrcJitter            → f23
  DstJitter            → f24
  SIntPkt              → f25
  DIntPkt              → f26
  Proto                → f27
  Dur                  → f28
  TcpRtt               → f29
  IdleTime             → f30
  Sum                  → f31
  Min                  → f32
  Max          

## Cell 3 — Fixed Sample Retrieval

No Chroma vector store exists for this dataset file.
Falls back to cosine-similarity sampling from the training dataframe.
Numeric columns only are used for similarity; strings are included in the output.

In [3]:
import numpy as np
import json
import ast
import re as _re
from sklearn.preprocessing import normalize

train_set_size = 100000
N_RETRIEVAL = 10


def _sample_via_dataframe(df, n: int) -> list:
    numeric_df = df.select_dtypes(include=[np.number])
    X = numeric_df.fillna(0).values.astype(float)
    if X.shape[1] == 0:
        return [str(df.iloc[i].to_list()) for i in np.random.choice(len(df), n, replace=False)]
    X_norm = normalize(X, norm='l2')
    mean_vec = X_norm.mean(axis=0)
    sims = X_norm @ mean_vec
    top_idx = np.argsort(sims)[-n:][::-1]
    return [str(df.iloc[i].to_list()) for i in top_idx]


def _quick_parse(doc):
    try:
        return json.loads(doc.replace("'", '"'))
    except Exception:
        pass
    try:
        return ast.literal_eval(doc)
    except Exception:
        pass
    tokens = _re.findall(
        r'[+-]?(?:nan|inf(?:inity)?|\d+\.?\d*(?:[eE][+-]?\d+)?)',
        doc, _re.IGNORECASE
    )
    return [float(t) for t in tokens]


try:
    from langchain_chroma import Chroma
    from langchain_huggingface.embeddings import HuggingFaceEmbeddings

    embeddings = HuggingFaceEmbeddings()
    vector_store = Chroma(
        collection_name=dataset_name,
        embedding_function=embeddings,
        persist_directory=f"./vector-stores/chroma-db-{train_set_size}-2"
    )
    normal_vectors = vector_store._collection.get(
        include=['embeddings'], where={'label': 'normal'})['embeddings']
    if not normal_vectors:
        raise ValueError("Vector store empty.")
    normal_mean_vec = np.mean(normal_vectors, axis=0).tolist()
    _raw_normal = vector_store._collection.query(
        query_embeddings=[normal_mean_vec], n_results=N_RETRIEVAL)['documents'][0]
    attack_vectors = vector_store._collection.get(
        include=['embeddings'], where={'label': 'attack'})['embeddings']
    attack_mean_vec = np.mean(attack_vectors, axis=0).tolist()
    _raw_attack = vector_store._collection.query(
        query_embeddings=[attack_mean_vec], n_results=N_RETRIEVAL)['documents'][0]
    vs_len = len(_quick_parse(_raw_normal[0]))
    if vs_len != len(feature_names):
        raise ValueError(f"Feature count mismatch: store={vs_len}, df={len(feature_names)}.")
    NORMAL_DOCS = _raw_normal
    ATTACK_DOCS = _raw_attack
    source = "vector store"
except Exception as e:
    print(f"⚠  {e}")
    print("   Sampling from dataframe.")
    NORMAL_DOCS = _sample_via_dataframe(normal_df_train, N_RETRIEVAL)
    ATTACK_DOCS = _sample_via_dataframe(attack_df_train, N_RETRIEVAL)
    source = "dataframe (cosine similarity to mean)"

print(f"✓  {len(NORMAL_DOCS)} normal + {len(ATTACK_DOCS)} attack docs | source: {source}")
print(f"   Values per doc: normal={len(_quick_parse(NORMAL_DOCS[0]))}, "
      f"attack={len(_quick_parse(ATTACK_DOCS[0]))}, expected={len(feature_names)}")
print("\nSample normal doc (first 120 chars):")
print(NORMAL_DOCS[0][:120], "...")

⚠  Vector store empty.
   Sampling from dataframe.
✓  10 normal + 10 attack docs | source: dataframe (cosine similarity to mean)
   Values per doc: normal=47, attack=47, expected=47

Sample normal doc (first 120 chars):
['2019-08-19 11:25:03', '2019-08-19 11:25:03', '192.168.0.44', '192.168.0.2', 0, 0, 0, 2, 2, 4, 124, 124, 248, 1053078.5 ...


/var/folders/zd/n69dqqnn6q561tsngmv534r80000gp/T/ipykernel_67582/324522671.py:51: DeprecationWarning: The truth value of an empty array is ambiguous. Returning False, but in future this will result in an error. Use `array.size > 0` to check that an array is not empty.
  if not normal_vectors:


## Cell 4 — Prompt Template

In [4]:
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

system_message = ("system",
"""
You are a good data analyst.
You are provided with network data entries categorized as either normal or attack, along with their corresponding feature names.
Carefully analyze the differences between normal and attack entries by comparing corresponding fields.
Your task is to generate {k} simple and deterministic rules for top {k} important features to filter attack entries.
Supported operators are '==', '!=', '>', '<', '>=', '<='.
Generate exactly {k} rules to filter attack entries and make a tool call for each rule.
"""
)

human_message = ("user",
"""
Analyze the following network data and generate rules for the top {k} important features to filter attack entries.

Normal Entries:
```{normal_entries}```

Attack Entries:
```{attack_entries}```
"""
)

prompt = ChatPromptTemplate.from_messages([
    system_message,
    human_message,
    MessagesPlaceholder("msgs")
])

print("Prompt template ready.")

Prompt template ready.


## Cell 5 — Anonymization-Aware evaluate_rule Tool + LLM

In [5]:
import operator
import os
import dotenv
import pandas as pd
from typing import Annotated
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI
from sklearn.metrics import classification_report

dotenv.load_dotenv(os.getcwd() + '/../.env')

OPERATIONS = {
    '<': operator.lt, '>': operator.gt,
    '==': operator.eq, '<=': operator.le,
    '>=': operator.ge, '!=': operator.ne,
}

@tool
def evaluate_rule(
    feature_name: Annotated[str, "Feature name (real name or f{i} anonymous label)"],
    value: Annotated[str, "Threshold value"],
    op: Annotated[str, "Comparison operator: ==, !=, >, <, >=, <="]
) -> float:
    """Evaluate a single threshold rule on the training set. Returns macro F1-score."""
    real_name = reverse_map.get(feature_name, feature_name)
    if op not in OPERATIONS:
        raise ValueError(f"Unsupported operator: {op}")
    op_fn = OPERATIONS[op]
    # dtype-aware coercion: string columns keep value as string
    sample_col = normal_df_train[real_name]
    if pd.api.types.is_numeric_dtype(sample_col):
        try:
            value = float(value)
        except (ValueError, TypeError):
            pass
    else:
        value = str(value)
    datasets = {"normal": normal_df_train, "attack": attack_df_train}
    y_pred, y_true = [], []
    for label, dataset in datasets.items():
        col = dataset[real_name].values
        preds = []
        for v in col:
            try:
                preds.append("attack" if op_fn(v, value) else "normal")
            except TypeError:
                preds.append("normal")
        y_pred.extend(preds)
        y_true.extend([label] * len(dataset))
    report = classification_report(y_true, y_pred, digits=4, output_dict=True,
                                   zero_division=0)
    return report['macro avg']['f1-score']


model_name = "gpt-4o"
llm = ChatOpenAI(model=model_name, temperature=0.1)
llm_with_tool = llm.bind_tools([evaluate_rule])

print(f"LLM: {model_name}, tool: evaluate_rule (anon-aware, dtype-aware)")

LLM: gpt-4o, tool: evaluate_rule (anon-aware, dtype-aware)


## Cell 6 — run_experiment() Function

In [6]:
import json
import ast
import re
import time
from langchain_core.messages import HumanMessage

try:
    from openai import RateLimitError
except ImportError:
    RateLimitError = Exception

N_ROUNDS = 5
K_RULES  = 5
INTER_ROUND_SLEEP = 5


def parse_doc(doc: str) -> list:
    try:
        return json.loads(doc.replace("'", '"'))
    except (json.JSONDecodeError, ValueError):
        pass
    try:
        return ast.literal_eval(doc)
    except (ValueError, SyntaxError):
        pass
    tokens = re.findall(
        r'[+-]?(?:nan|inf(?:inity)?|\d+\.?\d*(?:[eE][+-]?\d+)?)',
        doc, re.IGNORECASE
    )
    return [float(t) for t in tokens]


def build_entries(use_anonymized: bool) -> tuple:
    parsed_normal = [parse_doc(d) for d in NORMAL_DOCS]
    parsed_attack = [parse_doc(d) for d in ATTACK_DOCS]
    n_feats = len(feature_names)
    for j, p in enumerate(parsed_normal):
        if len(p) != n_feats:
            raise ValueError(f"Normal doc {j}: {len(p)} values, expected {n_feats}.")
    for j, p in enumerate(parsed_attack):
        if len(p) != n_feats:
            raise ValueError(f"Attack doc {j}: {len(p)} values, expected {n_feats}.")
    key_fn = (lambda i, name: f"f{i}") if use_anonymized else (lambda i, name: name)
    normal_entries = {key_fn(i, n): [p[i] for p in parsed_normal] for i, n in enumerate(feature_names)}
    attack_entries = {key_fn(i, n): [p[i] for p in parsed_attack] for i, n in enumerate(feature_names)}
    return normal_entries, attack_entries


def invoke_with_retry(chain, inputs, max_retries: int = 6, base_delay: float = 15.0):
    for attempt in range(max_retries):
        try:
            return chain.invoke(inputs)
        except RateLimitError:
            if attempt == max_retries - 1:
                raise
            wait = base_delay * (2 ** attempt)
            print(f"  ⚠ Rate limit (attempt {attempt+1}). Sleeping {wait:.0f}s...")
            time.sleep(wait)
        except Exception:
            raise


def run_experiment(use_anonymized: bool, verbose: bool = True) -> dict:
    condition = "anonymized" if use_anonymized else "original"
    normal_entries, attack_entries = build_entries(use_anonymized)
    chain = prompt | llm_with_tool
    n, mean_f1, msgs, train_f1s = 0, 0.0, [], []
    last_ai_msg, token_usage = None, {}
    while n < N_ROUNDS:
        ai_msg = invoke_with_retry(chain, {
            "k": K_RULES,
            "normal_entries": json.dumps(normal_entries),
            "attack_entries": json.dumps(attack_entries),
            "msgs": msgs,
        })
        last_ai_msg = ai_msg
        tool_msgs = [evaluate_rule.invoke(tc) for tc in ai_msg.tool_calls]
        mean_f1 = sum(float(m.content) for m in tool_msgs) / len(tool_msgs) if tool_msgs else 0.0
        train_f1s.append(mean_f1)
        feedback = HumanMessage(
            f"The current mean f1-score for the generated rules is {mean_f1:.4f}. "
            "If this mean f1-score is greater than the previous rounds, keep the better "
            "performing rules and revise or replace only the underperforming ones "
            "(those with a score less than mean). "
            "Otherwise, revise or replace any rules that have a score less than mean. "
            f"Based on the feedback, generate exactly {K_RULES} rules to filter attack "
            "entries and make a tool call for each rule, ensuring that a tool call is "
            "made for every entry every time."
        )
        msgs.extend([ai_msg, *tool_msgs, feedback])
        n += 1
        token_usage = ai_msg.response_metadata.get("token_usage", {})
        if verbose:
            print(f"  [{condition}] Round {n}/{N_ROUNDS}  "
                  f"mean_f1={mean_f1:.4f}  "
                  f"tokens={token_usage.get('total_tokens', '?')}")
        if n < N_ROUNDS:
            time.sleep(INTER_ROUND_SLEEP)
    return {
        "condition": condition,
        "train_f1s": train_f1s,
        "final_tool_calls": last_ai_msg.tool_calls,
        "all_msgs": msgs,
        "token_usage": token_usage,
    }


print("Parsing validation:")
for lbl, docs in [("normal", NORMAL_DOCS), ("attack", ATTACK_DOCS)]:
    lengths = {len(parse_doc(d)) for d in docs}
    ok = lengths == {len(feature_names)}
    print(f"  {lbl}: {len(docs)} docs, lengths={lengths}, expected={len(feature_names)}  {'✓' if ok else '⚠ MISMATCH'}")
print("run_experiment() ready.")

Parsing validation:
  normal: 10 docs, lengths={47}, expected=47  ✓
  attack: 10 docs, lengths={47}, expected=47  ✓
run_experiment() ready.


## Cell 7 — Multi-Seed Execution

In [7]:
import time

N_SEEDS          = 3
INTER_SEED_SLEEP = 60

print(f"=== Running ORIGINAL condition ({N_SEEDS} seeds) ===")
original_results = []
for seed in range(N_SEEDS):
    print(f"\n--- Original seed {seed+1}/{N_SEEDS} ---")
    result = run_experiment(use_anonymized=False, verbose=True)
    original_results.append(result)
    if seed < N_SEEDS - 1:
        print(f"  Sleeping {INTER_SEED_SLEEP}s...")
        time.sleep(INTER_SEED_SLEEP)

print(f"\n=== Running ANONYMIZED condition ({N_SEEDS} seeds) ===")
anon_results = []
for seed in range(N_SEEDS):
    print(f"\n--- Anonymized seed {seed+1}/{N_SEEDS} ---")
    result = run_experiment(use_anonymized=True, verbose=True)
    anon_results.append(result)
    if seed < N_SEEDS - 1:
        print(f"  Sleeping {INTER_SEED_SLEEP}s...")
        time.sleep(INTER_SEED_SLEEP)

print("\nAll runs complete.")

=== Running ORIGINAL condition (3 seeds) ===

--- Original seed 1/3 ---
  [original] Round 1/5  mean_f1=0.7315  tokens=5869
  [original] Round 2/5  mean_f1=0.8646  tokens=6643
  [original] Round 3/5  mean_f1=0.8314  tokens=7429
  [original] Round 4/5  mean_f1=0.7554  tokens=8191
  [original] Round 5/5  mean_f1=0.8162  tokens=8909
  Sleeping 60s...

--- Original seed 2/3 ---
  [original] Round 1/5  mean_f1=0.7315  tokens=5912
  [original] Round 2/5  mean_f1=0.6686  tokens=6608
  [original] Round 3/5  mean_f1=0.7315  tokens=7336
  [original] Round 4/5  mean_f1=0.7315  tokens=8054
  [original] Round 5/5  mean_f1=0.7315  tokens=8775
  Sleeping 60s...

--- Original seed 3/3 ---
  [original] Round 1/5  mean_f1=0.6443  tokens=5831
  [original] Round 2/5  mean_f1=0.7987  tokens=6456
  [original] Round 3/5  mean_f1=0.8349  tokens=7017
  [original] Round 4/5  mean_f1=0.8348  tokens=7579
  [original] Round 5/5  mean_f1=0.8348  tokens=8138

=== Running ANONYMIZED condition (3 seeds) ===

--- Anony

## Cell 8 — Test Set Evaluation

In [8]:
import pandas as pd
from statistics import mode
from sklearn.metrics import classification_report

def evaluate_rules_on_test(tool_calls: list) -> dict:
    if not tool_calls:
        print("  WARNING: empty tool_calls.")
        return {}
    datasets = {"normal": normal_df_test, "attack": attack_df_test}
    y_pred, y_true = [], []
    for label, dataset in datasets.items():
        for i in range(len(dataset)):
            row = dataset.iloc[i]
            predictions = []
            for tc in tool_calls:
                args      = tc["args"]
                fname     = args["feature_name"]
                real_name = reverse_map.get(fname, fname)
                op        = args["op"]
                val       = args["value"]
                sample_col = normal_df_test[real_name]
                if pd.api.types.is_numeric_dtype(sample_col):
                    try:
                        val = float(val)
                    except (ValueError, TypeError):
                        pass
                else:
                    val = str(val)
                if op in OPERATIONS:
                    try:
                        predictions.append(
                            "attack" if OPERATIONS[op](row[real_name], val) else "normal"
                        )
                    except TypeError:
                        predictions.append("normal")
            y_true.append(label)
            y_pred.append(mode(predictions) if predictions else "normal")
    return classification_report(y_true, y_pred, digits=4, output_dict=True,
                                 zero_division=0)


print("Evaluating original condition on test set...")
original_test_reports = [evaluate_rules_on_test(r["final_tool_calls"]) for r in original_results]
print("Evaluating anonymized condition on test set...")
anon_test_reports     = [evaluate_rules_on_test(r["final_tool_calls"]) for r in anon_results]
print("Done.")

Evaluating original condition on test set...
Evaluating anonymized condition on test set...
Done.


## Cell 9 — Comparison Metrics

In [9]:
from collections import Counter
import numpy as np
from tabulate import tabulate

orig_f1s = [r["macro avg"]["f1-score"] for r in original_test_reports]
anon_f1s = [r["macro avg"]["f1-score"] for r in anon_test_reports]
delta_f1 = np.mean(orig_f1s) - np.mean(anon_f1s)

def extract_features(tool_calls: list) -> set:
    features = set()
    for tc in tool_calls:
        fname = tc["args"]["feature_name"]
        features.add(reverse_map.get(fname, fname))
    return features

orig_features_per_run = [extract_features(r["final_tool_calls"]) for r in original_results]
anon_features_per_run = [extract_features(r["final_tool_calls"]) for r in anon_results]
orig_all_features = Counter(f for fs in orig_features_per_run for f in fs)
anon_all_features = Counter(f for fs in anon_features_per_run for f in fs)

def jaccard(a: set, b: set) -> float:
    return len(a & b) / len(a | b) if (a | b) else 0.0

jaccards     = [jaccard(o, a) for o, a in zip(orig_features_per_run, anon_features_per_run)]
mean_jaccard = np.mean(jaccards)
std_jaccard  = np.std(jaccards)

def category_dist(feature_sets: list) -> Counter:
    counts = Counter()
    for fs in feature_sets:
        for f in fs:
            counts[get_category(f)] += 1
    return counts

orig_cat = category_dist(orig_features_per_run)
anon_cat = category_dist(anon_features_per_run)

print("\n=== Per-Seed Feature Selection ===")
seed_rows = []
for i in range(N_SEEDS):
    seed_rows.append([
        i + 1,
        ", ".join(sorted(orig_features_per_run[i])),
        f"{orig_f1s[i]:.4f}",
        ", ".join(sorted(anon_features_per_run[i])),
        f"{anon_f1s[i]:.4f}",
        f"{jaccards[i]:.3f}",
    ])
print(tabulate(seed_rows,
    headers=["Seed", "Original features", "Orig F1",
             "Anon features (real names)", "Anon F1", "Jaccard"],
    tablefmt="grid"))

print("\n=== Condition Summary ===")
print(tabulate([
    ["Original",   f"{np.mean(orig_f1s):.4f}", f"{np.std(orig_f1s):.4f}"],
    ["Anonymized", f"{np.mean(anon_f1s):.4f}", f"{np.std(anon_f1s):.4f}"],
], headers=["Condition", "Mean Test F1", "Std"], tablefmt="grid"))
print(f"ΔF1 (original − anonymized): {delta_f1:+.4f}")
print(f"Mean Jaccard (feature overlap): {mean_jaccard:.3f} ± {std_jaccard:.3f}")

print("\n=== Semantic Category Distribution ===")
cat_rows = [
    [cat, orig_cat.get(cat, 0), anon_cat.get(cat, 0)]
    for cat in list(SEMANTIC_CATEGORIES.keys()) + ["unknown"]
    if orig_cat.get(cat, 0) > 0 or anon_cat.get(cat, 0) > 0
]
print(tabulate(cat_rows,
    headers=["Category", "Original (total)", "Anonymized (total)"],
    tablefmt="grid"))

print("\n=== Most-Selected Features — Original ===")
print(tabulate([[f, c] for f, c in orig_all_features.most_common()],
    headers=["Feature", f"Freq (/{N_SEEDS} seeds)"], tablefmt="grid"))

print("\n=== Most-Selected Features — Anonymized ===")
print(tabulate([[f, c] for f, c in anon_all_features.most_common()],
    headers=["Feature (real name)", f"Freq (/{N_SEEDS} seeds)"], tablefmt="grid"))


=== Per-Seed Feature Selection ===
+--------+-------------------------------------------+-----------+-----------------------------------------+-----------+-----------+
|   Seed | Original features                         |   Orig F1 | Anon features (real names)              |   Anon F1 |   Jaccard |
+========+===========================================+===========+=========================================+===========+===========+
|      1 | DstAddr, DstLoad, DstPkts, TotPkts, pLoss |    0.953  | Dport, Sport, SrcAddr, SrcLoad, SrcLoss |    0.6062 |      0    |
+--------+-------------------------------------------+-----------+-----------------------------------------+-----------+-----------+
|      2 | DstAddr, DstLoad, SrcAddr, SrcLoad, pLoss |    0.8134 | Dport, Sport, SrcAddr, SrcLoad, SrcLoss |    0.7318 |      0.25 |
+--------+-------------------------------------------+-----------+-----------------------------------------+-----------+-----------+
|      3 | DstLoad, DstPkts, Rate

## Cell 10 — Save Results

In [10]:
import json
import os

os.makedirs("results/anon", exist_ok=True)

def report_to_serializable(report: dict) -> dict:
    return {k: {mk: float(mv) for mk, mv in v.items()} if isinstance(v, dict) else float(v)
            for k, v in report.items()}

payload = {
    "dataset": dataset_name,
    "model": model_name,
    "n_seeds": N_SEEDS,
    "n_rounds": N_ROUNDS,
    "k_rules": K_RULES,
    "anon_map": anon_map,
    "original": {
        "test_f1_mean": float(np.mean(orig_f1s)),
        "test_f1_std":  float(np.std(orig_f1s)),
        "test_f1_per_seed": [float(f) for f in orig_f1s],
        "train_f1s_per_seed": [r["train_f1s"] for r in original_results],
        "selected_features_per_seed": [sorted(fs) for fs in orig_features_per_run],
        "feature_frequency": dict(orig_all_features),
        "category_dist": dict(orig_cat),
        "classification_reports": [report_to_serializable(r) for r in original_test_reports],
    },
    "anonymized": {
        "test_f1_mean": float(np.mean(anon_f1s)),
        "test_f1_std":  float(np.std(anon_f1s)),
        "test_f1_per_seed": [float(f) for f in anon_f1s],
        "train_f1s_per_seed": [r["train_f1s"] for r in anon_results],
        "selected_features_per_seed": [sorted(fs) for fs in anon_features_per_run],
        "feature_frequency": dict(anon_all_features),
        "category_dist": dict(anon_cat),
        "classification_reports": [report_to_serializable(r) for r in anon_test_reports],
    },
    "delta_f1": float(delta_f1),
    "mean_jaccard": float(mean_jaccard),
    "std_jaccard":  float(std_jaccard),
    "jaccards_per_seed": [float(j) for j in jaccards],
}

out_path = f"results/anon/{dataset_name}-anon-ablation.json"
with open(out_path, "w") as f:
    json.dump(payload, f, indent=2)

print(f"Results saved to: {out_path}")
print(f"\nKey findings:")
print(f"  Original  F1: {np.mean(orig_f1s):.4f} ± {np.std(orig_f1s):.4f}")
print(f"  Anonymized F1: {np.mean(anon_f1s):.4f} ± {np.std(anon_f1s):.4f}")
print(f"  ΔF1:           {delta_f1:+.4f}")
print(f"  Mean Jaccard:  {mean_jaccard:.3f} ± {std_jaccard:.3f}")

Results saved to: results/anon/wustl-iiot-anon-ablation.json

Key findings:
  Original  F1: 0.8458 ± 0.0778
  Anonymized F1: 0.6678 ± 0.0513
  ΔF1:           +0.1780
  Mean Jaccard:  0.083 ± 0.118
